# 載入多種資料來源

### 使用的官方 Loader（目前主流、穩定）

* PDF → `PyPDFLoader`
* TXT → `TextLoader`
* CSV → `CSVLoader`
* DOCX → `UnstructuredWordDocumentLoader`

這些都是 LangChain 官方文件與社群最常用、最穩定的組合。

---

### 需安裝的套件

#### 1. 基礎必裝

這一組可以支援：

- PyPDFLoader（PDF）
- TextLoader（TXT）
- CSVLoader（CSV）
- MergedDataLoader
- Document / BaseLoader 等核心抽象

```
pip install "unstructured[docx]"
```

#### 2. PDF 解析（目前用的是 PyPDFLoader）

```
pip install pypdf
```

#### 3. DOCX 解析

```
pip install "unstructured[docx]"
```

#### 如果想「一勞永逸」（不推薦但完整）

只有在你確定之後還會解析：
- HTML
- PPTX
- Image
- 複雜 PDF（含表格 / layout）

```
pip install "unstructured[all]"
```

注意：

> `[all]` 會拉一堆 system dependency，在 CI / Docker / Mac / Linux 都比較容易踩雷。

---

### 範例代碼

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional, Iterable, Dict
from collections import Counter

from langchain_core.documents import Document
from langchain_community.document_loaders.base import BaseLoader
from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    CSVLoader,
    UnstructuredWordDocumentLoader,
)
from langchain_community.document_loaders.merge import MergedDataLoader


# ------------------------------------------------------------
# 1) 文件類型常數（集中管理，避免 magic string）
# ------------------------------------------------------------
TECHNICAL_GUIDE = "technical_guide"
COMPANY_POLICY = "company_policy"
PRODUCT_CATALOG = "product_catalog"
API_DOCUMENTATION = "api_documentation"


# ------------------------------------------------------------
# 2) TaggedLoader：官方推薦的 ingestion 思路延伸
#    每個來源自己負責「語意打標」
#    重點價值：不改原 loader 邏輯，就能在 ingest 階段統一打標，方便後續檢索、過濾與追蹤來源。
# ------------------------------------------------------------
@dataclass(frozen=True)
class TaggedLoader(BaseLoader):
    loader: BaseLoader
    document_type: str
    source: Optional[str] = None

    def load(self) -> list[Document]:
        docs = self.loader.load()
        stamped: list[Document] = []

        for d in docs:
            # 先取得 document 的 metadata
            metadata = dict(d.metadata or {})
            # 打標，即加入資訊到 metadata
            metadata["document_type"] = self.document_type
            if self.source:
                metadata["source"] = self.source

            # 打標後建立新的 Document 物件，更新 page_content / metadata，全部串起來。
            # Document 物件預設其實只有 page_content
            stamped.append(
                Document(
                    page_content=d.page_content,
                    metadata=metadata,
                    id=getattr(d, "id", None),
                )
            )

        return stamped


# ------------------------------------------------------------
# 3) 輔助：統計文件類型（方便你驗證 ingestion 是否正確）
# ------------------------------------------------------------
def summarize_docs(docs: Iterable[Document]) -> Dict[str, int]:
    # Counter + generator expression
    counter = Counter(
        (d.metadata or {}).get("document_type", "unknown")
        for d in docs
    )
    return dict(counter)


# ------------------------------------------------------------
# 4) 主流程：多檔案型態 → 統一知識庫
# ------------------------------------------------------------
def main():
    loaders = [
        # SOLID 原則 PDF
        TaggedLoader(
            PyPDFLoader("../data/raw/solid_principles.pdf"),
            TECHNICAL_GUIDE,
            source="solid_principles.pdf",
        ),

        # 公司政策 TXT
        TaggedLoader(
            TextLoader("../data/raw/company_policy.txt", encoding="utf-8"),
            COMPANY_POLICY,
            source="company_policy.txt",
        ),

        # 產品型錄 CSV
        TaggedLoader(
            CSVLoader("../data/raw/products.csv"),
            PRODUCT_CATALOG,
            source="products.csv",
        ),

        # 設計模式 DOCX
        TaggedLoader(
            UnstructuredWordDocumentLoader(
                "../data/raw/simple_factory_vs_factory_method.docx"
            ),
            API_DOCUMENTATION,
            source="simple_factory_vs_factory_method.docx",
        ),
    ]

    merged_loader = MergedDataLoader(loaders=loaders)
    # 真正進行打標的地方
    all_documents = merged_loader.load()

    print(f"知識庫總計包含 {len(all_documents)} 個文件")
    print("\n=== 文件類型統計 ===")

    stats = summarize_docs(all_documents)
    for doc_type, count in sorted(stats.items()):
        print(f"{doc_type}: {count} 個文件")

    print("\n文件載入與整合完成！")


if __name__ == "__main__":
    main()

知識庫總計包含 21 個文件

=== 文件類型統計 ===
api_documentation: 1 個文件
company_policy: 1 個文件
product_catalog: 10 個文件
technical_guide: 9 個文件

文件載入與整合完成！


### 這樣寫「才真的符合原本範例的用意」

#### 1️⃣ **檔案型態 ≠ 文件語意**

PDF、TXT、CSV、DOCX 只是「載入技術」，
真正重要的是：

> 這份內容在「知識系統中扮演什麼角色？」

所以我們用 `document_type` 描述的是「語意責任」，不是檔案格式。

#### 2️⃣ **整合點不該是 list.extend，而是 ingestion boundary**

原本：

```python
all_documents.extend(docs)
```

現在：

* 每個 loader 自己完成 parsing + tagging
* 主流程只負責「合併」

這會讓你未來加：

* WebLoader
* DBLoader
* Jira / Confluence / GitHub Loader

完全不用動核心邏輯。

#### 3️⃣ **這已經是企業級 RAG ingestion 的最小骨架**

這一版已經天然支援：

* Retriever filter（只查 company_policy）
* Prompt routing（policy 問題不用 technical_guide）
* Evaluation slicing（不同 document_type 各自評分）
* Governance（某些 document_type 需要人工審核）

---


### 一句話 Takeaway（幫你定位這段 code）

> 多檔案型態 ingestion 的關鍵，不是「怎麼讀檔」，而是「你如何在 ingestion 階段就定義文件的語意責任」。

#### 你下一步可以立刻做的事（很實際）

下一步不是再加 loader，而是：

👉 在 retriever 加一個 `filter={"document_type": ...}`
👉 或在 prompt 裡顯式告訴 LLM：「回答此問題時只能使用 company_policy」

那一刻你會清楚感覺到：
**這段 code 不再是範例，而是系統設計的一部分。**

如果你要，我可以直接幫你接到一個最小 RAG pipeline（含 filter 與 prompt routing），讓這組資料「真的開始工作」。

---

### 解析網頁時是否應使用 unstructured[all] 套件而不是 webbaseloader？

結論先講：**「會不會比 WebBaseLoader 好？」取決於你要的是什麼。**
如果你要「快、穩、抓得到就好」→ WebBaseLoader 通常更合適。
如果你要「語意更乾淨、保留標題/段落/條列結構、後面 RAG 準確率更穩」→ 用 unstructured 解析 HTML 往往更好，但成本更高。

下面我把差異講得很具體，讓你一看就知道怎麼選。

---

你可以把兩者想成兩種層級的工具：

1. WebBaseLoader：網頁抓取 + 轉成文字
   WebBaseLoader 的定位是「把網頁 HTML 抓回來，提取文字塞進 Document」，適合快速 ingestion。官方文件也直接說它是用來載入 HTML 網頁文字，如果你想更省事（例如 crawling、JS 阻擋、清洗）還可以考慮其他 loader。([LangChain 文件][1])

2. unstructured（解析 HTML）：文件理解 + 元素切分
   LangChain 的 UnstructuredHTMLLoader（或直接用 unstructured）會把 HTML 分成「元素」，像 Title、NarrativeText、ListItem 這類，等於它在 ingestion 階段就先幫你做「語意邊界切分」。([LangChain 參考文檔][2])

---

實務上最大的差別其實只有三個：

第一個差別：輸出形狀（會直接影響 RAG 品質）
WebBaseLoader 通常會給你「一大坨文字」（或頁面為單位），你後面要靠 splitter 自己切。([LangChain 文件][1])
UnstructuredHTMLLoader 可以用 elements 模式，會把內容切成「標題一塊、段落一塊、條列一塊」，你後面 chunking 通常更穩。([LangChain 參考文檔][2])

第二個差別：雜訊與清洗
WebBaseLoader 很容易把導覽列、頁尾、推薦文章、cookie banner 一起抓進來，官方也暗示如果要更客製的邏輯可以看其他 child loader 或自己客製。([LangChain 文件][1])
Unstructured 對「把內容拆成語意元素」比較強，很多時候能讓你更容易只保留你要的元素（例如只留 NarrativeText / Title）。([docs.unstructured.io][3])
但要注意：它不是魔法，廣告或側欄照樣可能被當成元素，你仍然要做 filter。

第三個差別：成本（依賴、速度、穩定性）
WebBaseLoader 較輕、較快、依賴少。
unstructured[all] 很重、安裝與系統依賴多、速度慢，但語意更好（尤其你想做「元素級 chunk」時）。LangChain 也明確把 Unstructured 定位成「支援很多類型檔案，需要額外的 system dependencies」的方案。([LangChain 文件][4])

---

給你三個「清楚的選擇情境」，你照著套就不會猶豫：

情境 A：你要快速做 demo、抓幾篇文章做 RAG
選 WebBaseLoader。理由：摩擦最小，先跑起來，品質問題後面再補。([LangChain 文件][1])

情境 B：你抓的是「公司內部 wiki / SOP / 規範」，你很在意不要引用錯段
優先考慮 unstructured 的 elements 模式，因為語意邊界更乾淨，後面檢索比較穩。([LangChain 參考文檔][2])

情境 C：網站是 SPA / 需要 JS render / 反爬很重
兩者都可能卡。這時你通常要用「能處理渲染與爬取」的專用 loader（官方在 WebBaseLoader 頁也提到可以考慮 FireCrawlLoader / SpiderLoader 這類）。([LangChain 文件][1])
（重點：這不是解析器問題，是抓取層問題。）

---

我會給你一個很工程的建議：用「兩段式」最穩

1. 抓取層：先用 WebBaseLoader 或能處理 JS 的抓取工具，把 HTML 拿到手
2. 解析層：再把 HTML 丟給 unstructured 做 elements partition，再做你的 filter（只留 Title/NarrativeText/ListItem）

這樣你就不會把「抓不到頁面」跟「解析品質不好」混在一起 debug。

---

一句話 takeaway：
如果你要「少踩坑、先跑起來」用 WebBaseLoader；如果你要「語意結構乾淨、元素級 chunk」用 unstructured（但要付出依賴與速度成本）。([LangChain 文件][1])

如果你願意，我可以照你現在的 repo 結構，給你一個最小可用的 `load_web(url) -> List[Document]`：同時支援 WebBaseLoader 與 Unstructured 的 fallback，並且把 `document_type/source/url` 的 metadata 打好，直接接到你前面那套多來源 ingestion pipeline。

[1]: https://docs.langchain.com/oss/python/integrations/document_loaders/web_base?utm_source=chatgpt.com "WebBaseLoader integration"
[2]: https://reference.langchain.com/v0.3/python/community/document_loaders/langchain_community.document_loaders.html.UnstructuredHTMLLoader.html?utm_source=chatgpt.com "UnstructuredHTMLLoader — 🦜🔗 LangChain documentation"
[3]: https://docs.unstructured.io/open-source/core-functionality/partitioning?utm_source=chatgpt.com "Partitioning"
[4]: https://docs.langchain.com/oss/python/integrations/document_loaders/unstructured_file?utm_source=chatgpt.com "Unstructured integration"

